# Preprocesamiento de Intensidades de Test

Este notebook se encarga de cargar, limpiar y preparar el conjunto de datos de intensidades de test (`data/raw/intensidades_new_cobre.csv`) para su posterior consumo en la inferencia del sensor blando.

In [ ]:
import pandas as pd
import numpy as np
import os
import warnings
warnings.filterwarnings('ignore')

In [ ]:
# Determinar la ubicación de la carpeta de datos de forma dinámica
base_dir = '..' if os.path.exists('../data') else '.'
input_path = os.path.join(base_dir, 'data/raw/intensidades_new_cobre.csv')
output_dir = os.path.join(base_dir, 'data/processed')
output_path = os.path.join(output_dir, 'intensidades_clean_no_duplicates.csv')

# 1. Cargar el dataset de intensidades crudas de test
df_int = pd.read_csv(input_path)
print(f"Total registros crudos: {len(df_int)}")

# 2. Construir la columna 'timestamp' uniendo 'date' y 'time'
df_int['timestamp'] = pd.to_datetime(df_int['date'].astype(str) + ' ' + df_int['time'].astype(str), errors='coerce')

# 3. Ordenar cronológicamente y limpiar index
df_int = df_int.sort_values('timestamp').reset_index(drop=True)

# 4. Definir las variables espectrométricas de interés para el modelo
features_spectrometric = ['n1fe', 'n2cu', 'n3zn', 'n4mo', 'n6sc']

# 5. Conservar únicamente la primera aparición del cambio físico en las lecturas (eliminar duplicados consecutivos/repetidos)
df_int_clean = df_int.drop_duplicates(subset=features_spectrometric, keep='first').reset_index(drop=True)
print(f"Total registros únicos: {len(df_int_clean)}")

# 6. Seleccionar las columnas finales
df_output = df_int_clean[['timestamp'] + features_spectrometric]

# 7. Guardar el archivo limpio en la carpeta de procesados
os.makedirs(output_dir, exist_ok=True)
df_output.to_csv(output_path, index=False)
print(f"Dataset limpio guardado exitosamente en: {output_path}")